# 06 Experiment: Idiom-Aware MT Few-Shot Prompting with FLAN-T5

## Overview
This notebook evaluates **FLAN-T5** as a prompting-based comparison model for the English→German idiom translation project.

### Why emphasize idioms more than WMT?
The paper’s main question is whether **two-stage fine-tuning and LoRA** improve **idiom translation** while preserving general-domain quality. FLAN-T5 is included as a **prompted comparison point**, not as the paper’s primary method. Because of that:

- **Idioms** get the full prompting experiment: **0-shot, 3-shot, and 5-shot**
- **WMT14** is used as a **lighter general-domain check**, so we run **0-shot only** there

This keeps the notebook aligned with the paper’s main contribution: understanding **idiom-specific behavior** and the specialization/generalization tradeoff, rather than turning the project into a prompt-ablation study.

### What this notebook covers
- FLAN-T5 prompting on the idiom dataset
- 0-shot / 3-shot / 5-shot idiom experiments
- FLAN-T5 0-shot prompting on a canonical 25-example WMT14 subset
- BLEU / chrF logging and summary artifact updates


## Clone Repo / Mount Drive

In [1]:
# Optional: clone the repo fresh in Colab
!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls

Cloning into 'Idiom-Aware-Finetuning-in-MT-EN-to-DE'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 86 (delta 36), reused 34 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 215.01 KiB | 3.36 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/Idiom-Aware-Finetuning-in-MT-EN-to-DE
configs  notebooks  README.md  results	scripts  src


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Dependencies & Imports

In [3]:
!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.9 MB/s eta 0:00:00


In [4]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Experiment Configuration

In [5]:
MODEL_NAME = "google/flan-t5-small"
RUN_PREFIX = "flan_t5_small_prompt"
SHOT_COUNTS = [0, 3, 5]

PROJECT_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
PROMPT_RESULTS_DIR = RESULTS_DIR
os.makedirs(PROMPT_RESULTS_DIR, exist_ok=True)

GEN_KWARGS = {
    "max_new_tokens": 64,
    "do_sample": False,
    "num_beams": 4,
    "early_stopping": True,
}

BATCH_SIZE = 8
MAX_SOURCE_LEN = 512

# WMT extension config
WMT_RUN_PREFIX = f"{RUN_PREFIX}_0shot_wmt25"
N_WMT = 25
WMT_SAMPLE_SEED = 123

print("MODEL_NAME:", MODEL_NAME)
print("RUN_PREFIX:", RUN_PREFIX)
print("PROMPT_RESULTS_DIR:", PROMPT_RESULTS_DIR)
print("WMT_RUN_PREFIX:", WMT_RUN_PREFIX)


# Rerun controls
FORCE_RERUN_PROMPTS = False

# Canonical automatic-metrics file used downstream for clean analysis merges
CANONICAL_METRICS_PATH = os.path.join(RESULTS_DIR, "automatic_metrics_canonical.csv")

def infer_flan_model_label(run_name):
    run_name = str(run_name)
    if "0shot" in run_name:
        return "flan_t5_small_prompt_0shot"
    if "3shot" in run_name:
        return "flan_t5_small_prompt_3shot"
    if "5shot" in run_name:
        return "flan_t5_small_prompt_5shot"
    return run_name

def upsert_canonical_metric(model, split, metrics, *, source_notebook, source_run_name, notes=""):
    row = {
        "timestamp": pd.Timestamp.now(tz="America/Los_Angeles").isoformat(timespec="seconds"),
        "model": model,
        "model_label": infer_flan_model_label(model),
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "source_notebook": source_notebook,
        "source_run_name": source_run_name,
        "notes": notes,
    }

    df_new = pd.DataFrame([row])

    if os.path.exists(CANONICAL_METRICS_PATH):
        df_old = pd.read_csv(CANONICAL_METRICS_PATH)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    df["timestamp"] = df["timestamp"].astype(str)
    df = df.sort_values(["model_label", "split", "timestamp"])
    df = df.drop_duplicates(subset=["model_label", "split"], keep="last")
    df.to_csv(CANONICAL_METRICS_PATH, index=False)
    return df_new

def pred_file_exists(run_name, results_dir=PROMPT_RESULTS_DIR):
    return os.path.exists(os.path.join(results_dir, f"{run_name}_preds.csv"))


MODEL_NAME: google/flan-t5-small
RUN_PREFIX: flan_t5_small_prompt
PROMPT_RESULTS_DIR: /content/drive/MyDrive/ds266_idiom_mt/results
WMT_RUN_PREFIX: flan_t5_small_prompt_0shot_wmt25


## Section A: Idiom Prompting Experiments

This section runs the **core FLAN-T5 prompting experiments** on the idiom evaluation set. We keep **0-shot, 3-shot, and 5-shot** here because this is where FLAN is being evaluated most directly against the project’s main research question.

## Load Data

In [6]:
def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)
    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src", "tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

idiom_ds = load_idioms(seed=SEED)
print({k: len(v) for k, v in idiom_ds.items()})

README.md: 0.00B [00:00, ?B/s]

en-de.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'train': 800, 'dev': 100, 'test': 100}


In [7]:
idiom_train_df = idiom_ds["train"].to_pandas().copy()
idiom_dev_df = idiom_ds["test"].to_pandas().copy()
idiom_eval_df = idiom_ds["test"].to_pandas().copy()

print(idiom_train_df.head(2))
print(idiom_eval_df.head(2))

                                                 src  \
0  Radio promotion costs are ridiculously out of ...   
1       Are you hitting on me? You're a married man!   

                                                 tgt  
0  Die Kosten für Radiowerbung sind völlig aus de...  
1  Machst du mich an? Du bist ein verheirateter M...  
                                               src  \
0                Not a word of all this to anyone!   
1  I don't want to be the blind leading the blind.   

                                                 tgt  
0           Kein Wort von all dem zu irgendjemandem!  
1  Ich möchte nicht der Blinde sein, der die Blin...  


## Load General-Domain Data (WMT14)

In [8]:
def load_wmt14(ft_train_n=20000, ft_dev_n=2000, seed=42):
    wmt = load_dataset("wmt14", "de-en")  # contains train / validation / test

    def to_en_de(ex):
        tr = ex["translation"]
        return {"src": tr["en"], "tgt": tr["de"]}

    train = wmt["train"].map(to_en_de, remove_columns=wmt["train"].column_names)
    test = wmt["test"].map(to_en_de, remove_columns=wmt["test"].column_names)

    shuf = train.shuffle(seed=seed)
    ft_train = shuf.select(range(min(ft_train_n, len(shuf))))
    ft_dev = shuf.select(range(min(ft_train_n, len(shuf)), min(ft_train_n + ft_dev_n, len(shuf))))
    return DatasetDict({"ft_train": ft_train, "ft_dev": ft_dev, "wmt_test": test})

general_ds = load_wmt14(seed=SEED)
print({k: len(v) for k, v in general_ds.items()})


# Canonical WMT few-shot demo source for prompted evaluation
wmt_train_df = general_ds["ft_train"].to_pandas().copy()
print("WMT train demo pool rows:", len(wmt_train_df))


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Map:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Map:   0%|          | 0/3003 [00:00<?, ? examples/s]

{'ft_train': 20000, 'ft_dev': 2000, 'wmt_test': 3003}
WMT train demo pool rows: 20000


### WMT prompt demo pool

For WMT few-shot prompting, demo examples are sampled from `general_ds['ft_train']`, which is converted to `wmt_train_df`. This mirrors the idiom-side prompting setup, where demonstrations are sampled from the idiom training pool.

In [9]:
# Small sanity check for idiom prompts

## Prompt Helpers

In [10]:
def build_demo_block(example):
    return (
        "Translate the English sentence into natural German.\n\n"
        f"English: {example['src']}\n\n"
        f"German: {example['tgt']}"
    )

def sample_demos(train_df, n, exclude_src=None, seed=42):
    pool = train_df if exclude_src is None else train_df[train_df["src"] != exclude_src]
    if n == 0:
        return []
    return pool.sample(n=n, random_state=seed).to_dict("records")

def build_prompt(src_text, demos=None):
    demos = demos or []
    header = (
        "You are translating English idiomatic sentences into fluent, "
        "meaning-preserving German. Do not translate idioms word-for-word "
        "when a natural German rendering is more appropriate."
    )

    if len(demos) == 0:
        return (
            f"{header}\n\n"
            "Translate the English sentence into natural German.\n\n"
            f"English: {src_text}\n\n"
            "German:"
        )

    demo_text = "\n\n".join(build_demo_block(ex) for ex in demos)

    return (
        f"{header}\n\n"
        f"Examples:\n\n{demo_text}\n\n"
        "Now translate the next sentence.\n\n"
        f"English: {src_text}\n\n"
        "German:"
    )

In [11]:
example_row = idiom_eval_df.iloc[0]
example_demos = sample_demos(idiom_train_df, 3, exclude_src=example_row["src"], seed=SEED)
print(build_prompt(example_row["src"], example_demos)[:1600])

You are translating English idiomatic sentences into fluent, meaning-preserving German. Do not translate idioms word-for-word when a natural German rendering is more appropriate.

Examples:

Translate the English sentence into natural German.

English: Don't discount any job, see it as an opportunity to get a foot in the door.

German: Schlagen Sie keine Stelle aus, sondern sehen Sie sie als Chance, einen Fuß in die Tür zu bekommen.

Translate the English sentence into natural German.

English: Because I'm in the same boat.

German: Ich sitze nämlich im selben Boot.

Translate the English sentence into natural German.

English: Are you not biting off more than you can chew with this project?

German: Übernimmst du dich nicht mit diesem Projekt?

Now translate the next sentence.

English: Not a word of all this to anyone!

German:


## Load Prompted Model

In [12]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
try:
    model.generation_config.max_length = None
except Exception:
    pass
try:
    model.config.max_length = None
except Exception:
    pass
model.eval()
print("loaded:", MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

loaded: google/flan-t5-small


## Generation Helpers

In [13]:
@torch.no_grad()
def generate_texts(prompts, gen_kwargs=None, batch_size=BATCH_SIZE):
    gen_kwargs = gen_kwargs or GEN_KWARGS
    outputs = []

    import copy
    gen_cfg = copy.deepcopy(model.generation_config)
    try:
        gen_cfg.max_length = None
    except Exception:
        pass

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start : start + batch_size]
        enc = tok(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(device)

        generated = model.generate(
            **enc,
            generation_config=gen_cfg,
            **gen_kwargs,
        )
        decoded = tok.batch_decode(generated, skip_special_tokens=True)
        outputs.extend([d.strip() for d in decoded])

    return outputs

def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": bleu, "chrf": chrf}


In [14]:
# One-example generation sanity check
test_row = idiom_eval_df.iloc[0]
test_prompt = build_prompt(test_row["src"], demos=[])
print("PROMPT:\n")
print(test_prompt[:1200])

test_pred = generate_texts([test_prompt], gen_kwargs=GEN_KWARGS)
print("\nPREDICTION:\n")
print(test_pred[0])

print("\nREFERENCE:\n")
print(test_row["tgt"])

Passing `generation_config` together with generation-related arguments=({'num_beams', 'early_stopping', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


PROMPT:

You are translating English idiomatic sentences into fluent, meaning-preserving German. Do not translate idioms word-for-word when a natural German rendering is more appropriate.

Translate the English sentence into natural German.

English: Not a word of all this to anyone!

German:

PREDICTION:

Nicht ein Wortes zu jemanden!

REFERENCE:

Kein Wort von all dem zu irgendjemandem!


## Metrics Logger

In [15]:
PROMPT_METRICS_PATH = os.path.join(PROMPT_RESULTS_DIR, "prompting_metrics.csv")

def log_prompt_metrics(run_name, model_name, n_shots, split, metrics, n_eval, gen_kwargs, notes=""):
    row = {
        "timestamp": pd.Timestamp.now(tz="America/Los_Angeles").isoformat(timespec="seconds"),
        "run_name": run_name,
        "model_name": model_name,
        "n_shots": n_shots,
        "split": split,
        "bleu": metrics.get("bleu"),
        "chrf": metrics.get("chrf"),
        "n_eval": n_eval,
        "gen_kwargs": str(gen_kwargs),
        "notes": notes,
    }

    if os.path.exists(PROMPT_METRICS_PATH):
        existing = pd.read_csv(PROMPT_METRICS_PATH)
        out = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    else:
        out = pd.DataFrame([row])

    out.to_csv(PROMPT_METRICS_PATH, index=False)
    return row


## Run Zero-Shot and Few-Shot Prompting

In [16]:
all_run_summaries = []

for n_shots in SHOT_COUNTS:
    run_name = f"{RUN_PREFIX}_{n_shots}shot"
    pred_path = os.path.join(PROMPT_RESULTS_DIR, f"{run_name}_preds.csv")

    if pred_file_exists(run_name) and not FORCE_RERUN_PROMPTS:
        print(f"Loading existing predictions for {run_name} from: {pred_path}")
        pred_df = pd.read_csv(pred_path)
        prompts = pred_df["prompt"].tolist() if "prompt" in pred_df.columns else None
        preds = pred_df["pred"].tolist()
        refs = pred_df["tgt"].tolist() if "tgt" in pred_df.columns else idiom_eval_df["tgt"].tolist()
    else:
        prompts = []
        for idx, row in idiom_eval_df.iterrows():
            demos = sample_demos(idiom_train_df, n_shots, exclude_src=row["src"], seed=SEED + idx)
            prompts.append(build_prompt(row["src"], demos=demos))

        preds = generate_texts(prompts, gen_kwargs=GEN_KWARGS)
        refs = idiom_eval_df["tgt"].tolist()

        pred_df = idiom_eval_df.copy()
        pred_df["prompt"] = prompts
        pred_df["pred"] = preds
        pred_df["model_name"] = MODEL_NAME
        pred_df["n_shots"] = n_shots
        pred_df.to_csv(pred_path, index=False)

    metrics = compute_metrics(preds, refs)
    print(run_name, metrics)

    log_prompt_metrics(
        run_name=run_name,
        model_name=MODEL_NAME,
        n_shots=n_shots,
        split="idioms_test",
        metrics=metrics,
        n_eval=len(preds),
        gen_kwargs=GEN_KWARGS,
        notes="few-shot prompted baseline on idiom subset"
    )

    upsert_canonical_metric(
        run_name,
        "idioms_test",
        metrics,
        source_notebook="06_experiment_idiom_aware_MT_fewshot_prompting_flant5.ipynb",
        source_run_name=run_name,
        notes="few-shot prompted baseline on idiom subset"
    )

    all_run_summaries.append({
        "run_name": run_name,
        "model_name": MODEL_NAME,
        "n_shots": n_shots,
        **metrics,
        "pred_path": pred_path,
    })

summary_df = pd.DataFrame(all_run_summaries)
summary_df


flan_t5_small_prompt_0shot {'bleu': 8.126810293457321, 'chrf': 33.138795479524205}
flan_t5_small_prompt_3shot {'bleu': 7.769177973296203, 'chrf': 32.51758301307655}
flan_t5_small_prompt_5shot {'bleu': 7.623039483156583, 'chrf': 32.096459272914366}


,run_name,model_name,n_shots,bleu,chrf,pred_path
0,flan_t5_small_prompt_0shot,google/flan-t5-small,0,8.126810,33.138795,/content/drive/MyDrive/ds266_idiom_mt/results/...
1,flan_t5_small_prompt_3shot,google/flan-t5-small,3,7.769178,32.517583,/content/drive/MyDrive/ds266_idiom_mt/results/...
2,flan_t5_small_prompt_5shot,google/flan-t5-small,5,7.623039,32.096459,/content/drive/MyDrive/ds266_idiom_mt/results/...


## Inspect Sample Outputs

In [17]:
def load_pred_csv(run_name, results_dir=PROMPT_RESULTS_DIR):
    path = os.path.join(results_dir, f"{run_name}_preds.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing prediction file: {path}")
    return pd.read_csv(path)

def preview_predictions(pred_df, n=10):
    cols = [c for c in ["src", "tgt", "pred"] if c in pred_df.columns]
    display(pred_df[cols].head(n))

for n_shots in SHOT_COUNTS:
    run_name = f"{RUN_PREFIX}_{n_shots}shot"
    print(f"=== {run_name} ===")
    preview_predictions(load_pred_csv(run_name), n=5)

=== flan_t5_small_prompt_0shot ===


,src,tgt,pred
0,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,Nicht ein Wortes zu jemanden!
1,I don't want to be the blind leading the blind.,"Ich möchte nicht der Blinde sein, der die Blin...","Ich ne hoffe nicht zu sein, daß ich die Blind ..."
2,"Listen, Barbara Kopetski, we could play cat an...","Hören Sie, Barbara Kopetski, wir könnten die g...","Ich hoffe, Barbara Kopetski, dass ich es es es..."
3,"I don't want to alarm you, but I just heard fr...","Ich möchte Sie nicht beunruhigen, aber ich hab...","Ich hoffe nicht, daß Sie es ernst, aber ich hö..."
4,"I don't mean to rain on your parade, but I hav...","Ich möchte Ihnen nicht in die Suppe spucken, a...","Ich weiß nicht, daß ich auf Ihres Parades fahr..."


=== flan_t5_small_prompt_3shot ===


,src,tgt,pred
0,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,Nicht ein Wortes zu jemanden!
1,I don't want to be the blind leading the blind.,"Ich möchte nicht der Blinde sein, der die Blin...","Ich ne hoffe nicht zu sein, daß ich die Blind ..."
2,"Listen, Barbara Kopetski, we could play cat an...","Hören Sie, Barbara Kopetski, wir könnten die g...","Wie ich, Barbara Kopetski, muß es muß es muß e..."
3,"I don't want to alarm you, but I just heard fr...","Ich möchte Sie nicht beunruhigen, aber ich hab...","Ich möchte Sie nicht alarmieren, aber ich hört..."
4,"I don't mean to rain on your parade, but I hav...","Ich möchte Ihnen nicht in die Suppe spucken, a...","Ich hoffe nicht, daß Sie fahren, aber ich habe..."


=== flan_t5_small_prompt_5shot ===


,src,tgt,pred
0,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,Nicht ein Wortes zu jemanden!
1,I don't want to be the blind leading the blind.,"Ich möchte nicht der Blinde sein, der die Blin...","Ich hoffe nicht, daß ich die Augen führen."
2,"Listen, Barbara Kopetski, we could play cat an...","Hören Sie, Barbara Kopetski, wir könnten die g...","Wie ich, Barbara Kopetski, können wir es es es..."
3,"I don't want to alarm you, but I just heard fr...","Ich möchte Sie nicht beunruhigen, aber ich hab...","Ich möchte Sie nicht alarmieren, aber ich hört..."
4,"I don't mean to rain on your parade, but I hav...","Ich möchte Ihnen nicht in die Suppe spucken, a...","Ich weiß nicht, dass Sie auf Ihrer Feiertag fa..."


## Diagnostics

In [18]:
def diagnose_outputs(preds, label="run"):
    preds = ["" if pd.isna(p) else str(p) for p in preds]
    stripped = [p.strip() for p in preds]

    lengths = [len(p) for p in stripped]
    unique_ratio = len(set(stripped)) / max(len(stripped), 1)
    empty_count = sum(1 for p in stripped if len(p) == 0)

    print(f"=== Diagnostics: {label} ===")
    print("n_outputs:", len(stripped))
    print("avg_len:", round(float(np.mean(lengths)), 2) if lengths else 0)
    print("min_len:", int(np.min(lengths)) if lengths else 0)
    print("max_len:", int(np.max(lengths)) if lengths else 0)
    print("empty_count:", empty_count)
    print("unique_ratio:", round(unique_ratio, 4))
    print()

for n_shots in SHOT_COUNTS:
    run_name = f"{RUN_PREFIX}_{n_shots}shot"
    pred_df = load_pred_csv(run_name)
    diagnose_outputs(pred_df["pred"].tolist(), label=run_name)

=== Diagnostics: flan_t5_small_prompt_0shot ===
n_outputs: 100
avg_len: 85.45
min_len: 29
max_len: 286
empty_count: 0
unique_ratio: 1.0

=== Diagnostics: flan_t5_small_prompt_3shot ===
n_outputs: 100
avg_len: 85.42
min_len: 29
max_len: 273
empty_count: 0
unique_ratio: 1.0

=== Diagnostics: flan_t5_small_prompt_5shot ===
n_outputs: 100
avg_len: 87.0
min_len: 29
max_len: 260
empty_count: 0
unique_ratio: 1.0



In [19]:
def compare_prompt_runs(run_names, n=10):
    loaded = [load_pred_csv(rn) for rn in run_names]
    out = pd.DataFrame({
        "src": loaded[0]["src"].head(n),
        "ref": loaded[0]["tgt"].head(n),
    })
    for rn, df in zip(run_names, loaded):
        out[rn] = df["pred"].head(n)
    return out

display(compare_prompt_runs([f"{RUN_PREFIX}_{s}shot" for s in SHOT_COUNTS], n=10))

,src,ref,flan_t5_small_prompt_0shot,flan_t5_small_prompt_3shot,flan_t5_small_prompt_5shot
0,Not a word of all this to anyone!,Kein Wort von all dem zu irgendjemandem!,Nicht ein Wortes zu jemanden!,Nicht ein Wortes zu jemanden!,Nicht ein Wortes zu jemanden!
1,I don't want to be the blind leading the blind.,"Ich möchte nicht der Blinde sein, der die Blin...","Ich ne hoffe nicht zu sein, daß ich die Blind ...","Ich ne hoffe nicht zu sein, daß ich die Blind ...","Ich hoffe nicht, daß ich die Augen führen."
2,"Listen, Barbara Kopetski, we could play cat an...","Hören Sie, Barbara Kopetski, wir könnten die g...","Ich hoffe, Barbara Kopetski, dass ich es es es...","Wie ich, Barbara Kopetski, muß es muß es muß e...","Wie ich, Barbara Kopetski, können wir es es es..."
3,"I don't want to alarm you, but I just heard fr...","Ich möchte Sie nicht beunruhigen, aber ich hab...","Ich hoffe nicht, daß Sie es ernst, aber ich hö...","Ich möchte Sie nicht alarmieren, aber ich hört...","Ich möchte Sie nicht alarmieren, aber ich hört..."
4,"I don't mean to rain on your parade, but I hav...","Ich möchte Ihnen nicht in die Suppe spucken, a...","Ich weiß nicht, daß ich auf Ihres Parades fahr...","Ich hoffe nicht, daß Sie fahren, aber ich habe...","Ich weiß nicht, dass Sie auf Ihrer Feiertag fa..."
5,A whippersnapper is someone who is younger tha...,"Ein Grünschnabel ist jemand, der jünger ist al...","Ein Whippersnapper ist ein Person, die sich jü...","Ein Whippersnapper ist ein Person, die sich jü...","Ein Whippersnapper ist ein Person, die sich jü..."
6,He was like a kid in a candy store when he saw...,"Er war wie ein Kind in einem Süßwarenladen, al...","He war wie ein Kinder in einem Candy-Store, we...","Er war wie ein Kinder in einem Candy-Store, we...","Er war wie ein Kinder in einem Candy-Store, we..."
7,"I thought I could understand the theories, but...","Ich dachte, ich könnte die Theorien verstehen,...","Ich dachte, ich zu verstehen die Theorien, abe...","Ich dachte, ich zu verstehen die Theorien, abe...","Ich dachte, ich zu verstehen die Theorien, abe..."
8,All part of the plan - to wear us down.,Alles Teil des Plans - um uns zu zermürben.,Alle Teil des Planes - die wir uns fahren.,Alle Teil der Plan - die wir uns fahren.,Alle Teil des Planes - die wir uns fahren.
9,He's been working late with her every night th...,Er hat diese Woche jeden Abend mit ihr lange g...,Er arbeitet es spät mit er jedes Nacht diese W...,Er hat es bei dieser Woche erarbeitet - Ich we...,Er hat er mit er jedes Nacht dieses Wochenende...


## Save Summary Artifact

In [20]:
summary_path = os.path.join(PROMPT_RESULTS_DIR, f"{RUN_PREFIX}_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("saved summary:", summary_path)
summary_df

saved summary: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_summary.csv


,run_name,model_name,n_shots,bleu,chrf,pred_path
0,flan_t5_small_prompt_0shot,google/flan-t5-small,0,8.126810,33.138795,/content/drive/MyDrive/ds266_idiom_mt/results/...
1,flan_t5_small_prompt_3shot,google/flan-t5-small,3,7.769178,32.517583,/content/drive/MyDrive/ds266_idiom_mt/results/...
2,flan_t5_small_prompt_5shot,google/flan-t5-small,5,7.623039,32.096459,/content/drive/MyDrive/ds266_idiom_mt/results/...


## Section B: WMT14 Prompting (General-Domain Baseline)

This section runs a **0-shot-only** FLAN-T5 experiment on a canonical **25-example WMT14 subset**.

We intentionally keep WMT lighter than the idiom side because WMT is serving here as a **general-domain reference point**, not as the main FLAN study.

In [21]:
def sample_examples(ds, n, seed=42, group=""):
    rng = random.Random(seed)
    idxs = list(range(len(ds)))
    rng.shuffle(idxs)
    idxs = idxs[:n]
    rows = []
    for i in idxs:
        rows.append({
            "group": group,
            "src": ds[i]["src"],
            "tgt": ds[i]["tgt"],
            "source_index": i,
        })
    return pd.DataFrame(rows)

wmt_subset_df = sample_examples(
    general_ds["wmt_test"],
    N_WMT,
    seed=WMT_SAMPLE_SEED,
    group="wmt_test",
)

print("Selected WMT subset rows:", len(wmt_subset_df))
display(wmt_subset_df.head())

Selected WMT subset rows: 25


,group,src,tgt,source_index
0,wmt_test,"And: ""In our society we have no major crimes, ...","Und: ""In unserer Gesellschaft gibt es keine Sc...",1165
1,wmt_test,"During Obama's transition to office in 2008, h...","Als damals 2008 Obama ins Amt wechselte, hatte...",2415
2,wmt_test,All three face a maximum penalty of life impri...,Allen dreien droht als Höchststrafe lebenslang...,186
3,wmt_test,Uwe Link has an offer for anyone who wants to ...,"Wer dann mit der Kutsche vorfahren will, für d...",1238
4,wmt_test,"First of all, US investment bank Goldman Sachs...",Zuerst hatte die US-Investmentbank Goldman Sac...,2304


## Run FLAN-T5 0-shot on WMT

In [22]:
wmt_run_summaries = []

for n_shots in SHOT_COUNTS:
    wmt_run_name = f"{RUN_PREFIX}_{n_shots}shot_wmt25"
    wmt_pred_path = os.path.join(PROMPT_RESULTS_DIR, f"{wmt_run_name}_preds.csv")
    wmt_subset_path = os.path.join(PROMPT_RESULTS_DIR, f"{wmt_run_name}_selected_subset.csv")

    if os.path.exists(wmt_pred_path) and os.path.exists(wmt_subset_path) and not FORCE_RERUN_PROMPTS:
        print(f"Loading existing WMT predictions for {wmt_run_name} from: {wmt_pred_path}")
        wmt_pred_df = pd.read_csv(wmt_pred_path)
        wmt_subset_df = pd.read_csv(wmt_subset_path)
        wmt_preds = wmt_pred_df["pred"].tolist()
        wmt_refs = wmt_pred_df["tgt"].tolist() if "tgt" in wmt_pred_df.columns else wmt_subset_df["tgt"].tolist()
    else:
        wmt_prompts = []
        for idx, row in wmt_subset_df.iterrows():
            demos = sample_demos(wmt_train_df, n_shots, exclude_src=row["src"], seed=WMT_SAMPLE_SEED + idx) if n_shots > 0 else []
            wmt_prompts.append(build_prompt(row["src"], demos=demos))

        wmt_preds = generate_texts(wmt_prompts, gen_kwargs=GEN_KWARGS)
        wmt_refs = wmt_subset_df["tgt"].tolist()

        wmt_pred_df = wmt_subset_df.copy()
        wmt_pred_df["prompt"] = wmt_prompts
        wmt_pred_df["pred"] = wmt_preds
        wmt_pred_df["model_name"] = MODEL_NAME
        wmt_pred_df["n_shots"] = n_shots

        wmt_pred_df.to_csv(wmt_pred_path, index=False)
        wmt_subset_df.to_csv(wmt_subset_path, index=False)

    wmt_metrics = compute_metrics(wmt_preds, wmt_refs)
    print(wmt_run_name, wmt_metrics)

    log_prompt_metrics(
        run_name=wmt_run_name,
        model_name=MODEL_NAME,
        n_shots=n_shots,
        split="wmt_test",
        metrics=wmt_metrics,
        n_eval=len(wmt_preds),
        gen_kwargs=GEN_KWARGS,
        notes=f"{n_shots}-shot prompted baseline on canonical 25-example WMT14 subset"
    )

    upsert_canonical_metric(
        f"{RUN_PREFIX}_{n_shots}shot",
        "wmt_test",
        wmt_metrics,
        source_notebook="06_experiment_idiom_aware_MT_fewshot_prompting_flant5.ipynb",
        source_run_name=wmt_run_name,
        notes=f"{n_shots}-shot prompted baseline on canonical 25-example WMT14 subset"
    )

    wmt_run_summaries.append({
        "run_name": wmt_run_name,
        "model_name": MODEL_NAME,
        "n_shots": n_shots,
        "bleu": wmt_metrics["bleu"],
        "chrf": wmt_metrics["chrf"],
        "pred_path": wmt_pred_path,
    })

wmt_summary_df = pd.DataFrame(wmt_run_summaries)
wmt_summary_df


flan_t5_small_prompt_0shot_wmt25 {'bleu': 7.615427563841016, 'chrf': 35.542535498334885}
flan_t5_small_prompt_3shot_wmt25 {'bleu': 7.431623079796857, 'chrf': 36.22497346319079}
flan_t5_small_prompt_5shot_wmt25 {'bleu': 4.958041805012363, 'chrf': 29.858081430308292}


,run_name,model_name,n_shots,bleu,chrf,pred_path
0,flan_t5_small_prompt_0shot_wmt25,google/flan-t5-small,0,7.615428,35.542535,/content/drive/MyDrive/ds266_idiom_mt/results/...
1,flan_t5_small_prompt_3shot_wmt25,google/flan-t5-small,3,7.431623,36.224973,/content/drive/MyDrive/ds266_idiom_mt/results/...
2,flan_t5_small_prompt_5shot_wmt25,google/flan-t5-small,5,4.958042,29.858081,/content/drive/MyDrive/ds266_idiom_mt/results/...


## Inspect WMT Outputs

In [23]:
display(wmt_pred_df[["src", "tgt", "pred"]].head(10))

,src,tgt,pred
0,"And: ""In our society we have no major crimes, ...","Und: ""In unserer Gesellschaft gibt es keine Sc...","Und: ""In unserem Gesellschaft haben wir keine ..."
1,"During Obama's transition to office in 2008, h...","Als damals 2008 Obama ins Amt wechselte, hatte...",In 2008 hat Obama eine 82% Abstimmungsranking.
2,All three face a maximum penalty of life impri...,Allen dreien droht als Höchststrafe lebenslang...,Alle drei treffen eine maximale Einwanderung v...
3,Uwe Link has an offer for anyone who wants to ...,"Wer dann mit der Kutsche vorfahren will, für d...","Die Siebendächerhaus, die sie in diesem Tagen ..."
4,"First of all, US investment bank Goldman Sachs...",Zuerst hatte die US-Investmentbank Goldman Sac...,Erstens hat die US Investment Bank Goldman Sac...
5,This became clear upon presentation of the ann...,Dies wurde bei der Vorstellung der Jahresrechn...,Dies hat deutlich über die Vorlage der jährlic...
6,She was right in the middle of the cliff face ...,Sie befand sich in der Mitte der Felswand – 15...,Umsatz- und Ertragss
7,"Meanwhile, U.S. lawmakers will head to Europe ...",Unterdessen werden US-amerikanische Gesetzgebe...,Gleichzeitig werden die U.S. Parlamenten in Eu...
8,The Observatory said there were casualties on ...,"Die Beobachtungsstelle sagte, es habe am Donne...","Die Observateur hat gesagt, daß die Opfer auf ..."
9,"Renamo wanted to ""warn the international commu...",Renamo wollte „die internationale Gemeinschaft...,"Herr Mazanga wollte es ""zuhalten die internati..."


## WMT Diagnostics

In [24]:
for _, row in wmt_summary_df.iterrows():
    pred_df = pd.read_csv(row["pred_path"])
    diagnose_outputs(pred_df["pred"].tolist(), label=row["run_name"])


=== Diagnostics: flan_t5_small_prompt_0shot_wmt25 ===
n_outputs: 25
avg_len: 105.6
min_len: 1
max_len: 194
empty_count: 0
unique_ratio: 1.0

=== Diagnostics: flan_t5_small_prompt_3shot_wmt25 ===
n_outputs: 25
avg_len: 115.04
min_len: 1
max_len: 278
empty_count: 0
unique_ratio: 1.0

=== Diagnostics: flan_t5_small_prompt_5shot_wmt25 ===
n_outputs: 25
avg_len: 106.52
min_len: 1
max_len: 278
empty_count: 0
unique_ratio: 1.0



## Update Summary Artifact

In [25]:
summary_plus_wmt = pd.concat(
    [
        summary_df.copy(),
        wmt_summary_df.rename(columns={"run_name": "run_name", "model_name": "model_name", "n_shots": "n_shots"}),
    ],
    ignore_index=True,
)

summary_plus_wmt_path = os.path.join(PROMPT_RESULTS_DIR, f"{RUN_PREFIX}_summary_with_wmt.csv")
summary_plus_wmt.to_csv(summary_plus_wmt_path, index=False)

print("saved extended summary:", summary_plus_wmt_path)
summary_plus_wmt


saved extended summary: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_summary_with_wmt.csv


,run_name,model_name,n_shots,bleu,chrf,pred_path
0,flan_t5_small_prompt_0shot,google/flan-t5-small,0,8.126810,33.138795,/content/drive/MyDrive/ds266_idiom_mt/results/...
1,flan_t5_small_prompt_3shot,google/flan-t5-small,3,7.769178,32.517583,/content/drive/MyDrive/ds266_idiom_mt/results/...
2,flan_t5_small_prompt_5shot,google/flan-t5-small,5,7.623039,32.096459,/content/drive/MyDrive/ds266_idiom_mt/results/...
3,flan_t5_small_prompt_0shot_wmt25,google/flan-t5-small,0,7.615428,35.542535,/content/drive/MyDrive/ds266_idiom_mt/results/...
4,flan_t5_small_prompt_3shot_wmt25,google/flan-t5-small,3,7.431623,36.224973,/content/drive/MyDrive/ds266_idiom_mt/results/...
5,flan_t5_small_prompt_5shot_wmt25,google/flan-t5-small,5,4.958042,29.858081,/content/drive/MyDrive/ds266_idiom_mt/results/...


### Canonical metrics quick check

This notebook does not train FLAN-T5. It performs prompted inference only, so there are no model checkpoints to save. The critical outputs to verify are prediction CSVs, historical prompting metrics, and the canonical automatic-metrics rows.

In [26]:
if os.path.exists(CANONICAL_METRICS_PATH):
    canonical_df = pd.read_csv(CANONICAL_METRICS_PATH)
    display(
        canonical_df[
            canonical_df["model_label"].isin([
                "flan_t5_small_prompt_0shot",
                "flan_t5_small_prompt_3shot",
                "flan_t5_small_prompt_5shot",
            ])
        ].sort_values(["model_label", "split"])
    )
    print("Canonical file:", CANONICAL_METRICS_PATH)
else:
    print("Canonical file not found yet:", CANONICAL_METRICS_PATH)


,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
2,2026-03-29T12:33:08-07:00,flan_t5_small_prompt_0shot,flan_t5_small_prompt_0shot,idioms_test,8.126810,33.138795,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_0shot,few-shot prompted baseline on idiom subset
3,2026-03-29T12:39:15-07:00,flan_t5_small_prompt_0shot,flan_t5_small_prompt_0shot,wmt_test,7.615428,35.542535,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_0shot_wmt25,0-shot prompted baseline on canonical 25-examp...
4,2026-03-29T12:35:43-07:00,flan_t5_small_prompt_3shot,flan_t5_small_prompt_3shot,idioms_test,7.769178,32.517583,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_3shot,few-shot prompted baseline on idiom subset
5,2026-03-29T12:40:10-07:00,flan_t5_small_prompt_3shot,flan_t5_small_prompt_3shot,wmt_test,7.431623,36.224973,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_3shot_wmt25,3-shot prompted baseline on canonical 25-examp...
6,2026-03-29T12:38:49-07:00,flan_t5_small_prompt_5shot,flan_t5_small_prompt_5shot,idioms_test,7.623039,32.096459,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_5shot,few-shot prompted baseline on idiom subset
7,2026-03-29T12:41:12-07:00,flan_t5_small_prompt_5shot,flan_t5_small_prompt_5shot,wmt_test,4.958042,29.858081,06_experiment_idiom_aware_MT_fewshot_prompting...,flan_t5_small_prompt_5shot_wmt25,5-shot prompted baseline on canonical 25-examp...


Canonical file: /content/drive/MyDrive/ds266_idiom_mt/results/automatic_metrics_canonical.csv


### Prediction artifact quick check

In [27]:
expected_pred_files = [os.path.join(PROMPT_RESULTS_DIR, f"{RUN_PREFIX}_{n}shot_preds.csv") for n in SHOT_COUNTS]
expected_pred_files += [os.path.join(PROMPT_RESULTS_DIR, f"{RUN_PREFIX}_{n}shot_wmt25_preds.csv") for n in SHOT_COUNTS]

for path in expected_pred_files:
    print(path, "exists:", os.path.exists(path))


/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_0shot_preds.csv exists: True
/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_3shot_preds.csv exists: True
/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_5shot_preds.csv exists: True
/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_0shot_wmt25_preds.csv exists: True
/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_3shot_wmt25_preds.csv exists: True
/content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_5shot_wmt25_preds.csv exists: True
